In [40]:
import os
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain.messages import SystemMessage, HumanMessage
from pydantic import RootModel
load_dotenv()

class Names(RootModel[dict[int, str]]):
    pass

def translate_names(names: list(tuple(int, str))) -> dict[int, str]:
    model = ChatMistralAI(
        api_key=os.getenv("MISTRAL_API_KEY"),
        model='mistral-medium-2505',
        max_retries=5
    )
    model = model.with_structured_output(Names)
    human_message = "Input: {stuff}"
    template = human_message.format(stuff=names)
    messages = [
        SystemMessage(content=
        "You are an Arabic to English translator."
        "Your task is to double check the translation of the given Arabic music artist names to English."
        "You will receive a list of tuples in the form (id, original_name, english_name),"
        "original_name is the name in arabic and sometimes english, this is the name you will use"
        "english_name is the already put name, this is the one you will replace"
        "you will return the answer in a dict where the key is the id and the value is the translated ENGLISH name."
        "if the name appears correct to you, dont add it to the dict"
        ),
        HumanMessage(content=template)
    ]

    response = model.invoke(messages)
    return response.root
    

In [41]:
import sys
# sys.path.append('..')
from data_extraction.db_operations.get_features import execute
results = execute('select id, name, name_en from artists')
# print(results)

In [49]:
test_in_batches = []
batch_size = 10
# full_response = {}
from tqdm import tqdm
from data_extraction.utils.rate_limit_handling import RateLimiter
llm_limiter = RateLimiter(0.38) 
for i in tqdm(range(10, len(results), batch_size)):
    batch = results[i:i + batch_size]
    llm_limiter.wait()
    try:
        print("trying batch", batch)
        full_response.update(translate_names(batch))
    except Exception as e:
        print(e)
import json
with open("name_en_names_revised.json", "w") as f:
    json.dump(full_response, f)

  1%|          | 1/109 [00:00<00:59,  1.82it/s]

1 validation error for Names
  Input should be a valid dictionary [type=dict_type, input_value=PydanticUndefined, input_type=PydanticUndefinedType]
    For further information visit https://errors.pydantic.dev/2.13/v/dict_type


  6%|▋         | 7/109 [00:16<04:05,  2.41s/it]

1 validation error for Names
  Input should be a valid dictionary [type=dict_type, input_value=PydanticUndefined, input_type=PydanticUndefinedType]
    For further information visit https://errors.pydantic.dev/2.13/v/dict_type


  7%|▋         | 8/109 [00:18<04:09,  2.47s/it]

...

 61%|██████▏   | 67/109 [02:55<01:38,  2.35s/it]

1 validation error for Names
  Input should be a valid dictionary [type=dict_type, input_value=PydanticUndefined, input_type=PydanticUndefinedType]
    For further information visit https://errors.pydantic.dev/2.13/v/dict_type


 66%|██████▌   | 72/109 [03:08<01:28,  2.39s/it]

1 validation error for Names
  Input should be a valid dictionary [type=dict_type, input_value=PydanticUndefined, input_type=PydanticUndefinedType]
    For further information visit https://errors.pydantic.dev/2.13/v/dict_type


100%|██████████| 109/109 [04:47<00:00,  2.63s/it]


In [63]:
import json

# Load IDs and revised English names from name_en_names_revised.json
with open("name_en_names_revised.json", "r", encoding="utf-8") as f:
    name_en_names_revised = json.load(f)

revised_ids = set(name_en_names_revised.keys())

# Assume execute() is already imported and works like: execute("select id, name, name_en from artists")
artist_data = execute("select id, name, name_en from artists")

# Build dict using the ENGLISH NAME from name_en_names_revised.json, NOT from the DB!
filtered_artists_dict = {}
for row in artist_data:
    id_ = str(row[0])  # IDs from JSON are strings
    if id_ in revised_ids:
        name = row[1]
        # use the EN name from name_en_names_revised, NOT from DB
        name_en = name_en_names_revised[id_]
        filtered_artists_dict[id_] = {"name": name, "name_en": name_en}

# Optionally, dump to file:
with open("filtered_artists_names_dump.json", "w", encoding="utf-8") as f:
    json.dump(filtered_artists_dict, f, ensure_ascii=False, indent=2)
len(filtered_artists_dict)

392

In [1]:
import sys
sys.path.append("..")
from translate_names.main import translate_names
from data_extraction.db_operations.get_features import execute
import json
with open("temp_response.json", "r") as f:
    full_response = json.load(f)
results = execute("select id, name, name_en from artists;")
test_in_batches = []
batch_size = 100
# full_response = {}
from time import sleep
from tqdm import tqdm
for i in tqdm(range(1, len(results), batch_size)):
    batch = results[i:i + batch_size]
    full_response.update(translate_names(batch))
    sleep(5)
import json
with open("name_en_names.json", "w") as f:
    json.dump(full_response, f)

python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 8
100%|██████████| 12/12 [04:24<00:00, 22.08s/it]


In [7]:
from translate_names.main import translate_names
response = translate_names(test__in)
response

python-dotenv could not parse statement starting at line 8


{1: 'Amr Diab',
 2: 'Fairuz',
 3: 'Om Kolthoum',
 4: 'Mohamed Mounir',
 5: 'Nancy Ajram',
 6: 'Marcel Khalifé',
 7: 'Hussain Al Jassmi',
 8: 'Cairokee',
 9: 'Balti',
 10: 'Khaled'}